In [2]:
import pathlib as pl
import pickle as pck
import pandas as pd

cfg_nb = pl.Path("../../load-config.ipynb").resolve(strict=True)
%run $cfg_nb

_NB_SESSION = __session__
NB_NAME = pl.Path(_NB_SESSION).name
NB_PATH = pl.Path(_NB_SESSION).parent
NB_REL_PATH = NB_PATH.relative_to(CONFIG["project_repo"]).joinpath(NB_NAME)

sample_sheet_file_assm = CONFIG["project_repo"].joinpath("samples", "verkko_chrY_arhie_freeze.tsv")
sample_sheet_assm = pd.read_csv(sample_sheet_file_assm, sep="\t", header=0, comment="#")

flagger_path = CONFIG["local_hilbert_prefix"].joinpath(
    CONFIG["flagger_annotation"]
).resolve(strict=True)

nucflag_path = CONFIG["local_hilbert_prefix"].joinpath(
    CONFIG["nucflag_annotation"]
).resolve(strict=True)


def find_all_bed_files(folder):

    bed_files = list(folder.glob("**/*.bed"))
    if not bed_files:
        raise FileNotFoundError(f"No bed files id'd: {folder}")
    return sorted(bed_files)
        

def extract_sample_id(qc_files):

    sample_file = []
    for filepath in qc_files:
        filename = filepath.name
        if "nucflag" in filename:
            sample = filename.split(".")[0]
        elif "flagger" in str(filepath):
            sample = filename.split(".")[0]
        else:
            raise ValueError(filepath)
        sample_file.append((sample, replace_path_prefix(filepath, local_to_remote=True)))
    return sample_file


FORCE_REBUILD_CACHE = True

cache_file_path = prep_cache_file(NB_REL_PATH, "qc_annot_sex_chrom.pkl")
if not cache_file_path.is_file() or FORCE_REBUILD_CACHE:

    qc_files = {
        "nucflag": find_all_bed_files(nucflag_path),
        "flagger": find_all_bed_files(flagger_path)
    }

    with open(cache_file_path, "wb") as cache:
        _ = pck.dump(qc_files, cache)
else:
    with open(cache_file_path, "rb") as cache:
        qc_files = pck.load(cache)
    
qc_files = {
    "nucflag": dict(extract_sample_id(qc_files["nucflag"])),
    "flagger": dict(extract_sample_id(qc_files["flagger"]))
}

sample_sheet_assm["nucflag"] = sample_sheet_assm["sample"].replace(qc_files["nucflag"])
sample_sheet_assm["flagger"] = sample_sheet_assm["sample"].replace(qc_files["flagger"])

# debug while waiting for remaining flagger output
keep_samples = sample_sheet_assm["flagger"].str.endswith(".bed")
sample_sheet_assm = sample_sheet_assm.loc[keep_samples, :].copy()
sample_sheet_assm.set_index("sample", inplace=True)

# decision from 2025-11-18
# drop sample HG03456 because of XYY karyotype
sample_sheet_assm.drop("HG03456

sample_sheet_file = CONFIG["project_repo"].joinpath("samples", "verkko_chrY_arhie_freeze.assm-qc.tsv")
with open(sample_sheet_file, "w") as table:
    _ = table.write(f"# {TIMESTAMP}\n")
    _ = table.write(f"# N={sample_sheet_assm.shape[0]}\n")
    for group, count in sample_sheet_assm["sample_group"].value_counts().items():
        _ = table.write(f"# group {group}: N={count}\n")
    sample_sheet_assm.to_csv(table, sep="\t", header=True, index=True)
